<a href="https://colab.research.google.com/github/Delean-Mafra/faculdade/blob/main/Arquitetura-de-BI-e-Big-Data-com-Ciencia-de-Dados-Aplicada-main/atividade14/Atividade_Pratica_14_versao_beta_teste.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Atividade Prática 14 – Bancos de Dados Distribuídos e Big Data

Este notebook contém a **solução consolidada e unificada** para a atividade, integrando:
*   **Ambiente NoSQL:** Elasticsearch para buscas e armazenamento.
*   **Processamento de Big Data:** Apache Spark para análise de faturamento.
*   **Inteligência de Dados:** Sistema de Recomendação baseado em co-ocorrência de compras.
*   **Monitoramento:** Prometheus para métricas de execução.

In [ ]:
import os
import time
import random
import csv
from datetime import datetime, timedelta
from collections import defaultdict
from elasticsearch import Elasticsearch, helpers
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as _sum
from prometheus_client import start_http_server, Counter

# --- 1. CONFIGURAÇÃO E INSTALAÇÃO ---
print("Instalando dependências...")
!pip install -q pyspark prometheus_client elasticsearch==6.8.2

# Setup do Elasticsearch 6.8.23 (Versão compatível com bypass no Colab)
ES_PATH = "/content/elasticsearch-6.8.23"
if not os.path.exists(ES_PATH):
    print("Baixando Elasticsearch...")
    !wget -q https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-6.8.23.tar.gz
    !tar -xzf elasticsearch-6.8.23.tar.gz
    !rm elasticsearch-6.8.23.tar.gz

# Limpeza de processos e permissões para o usuário 'daemon'
!pkill -9 -f elasticsearch
!rm -rf {ES_PATH}/data {ES_PATH}/logs
!mkdir -p {ES_PATH}/data {ES_PATH}/logs
!chown -R daemon:daemon {ES_PATH}
!chmod -R 777 {ES_PATH}

# Configuração para rodar em nó único
with open(f'{ES_PATH}/config/elasticsearch.yml', 'w') as f:
    f.write("cluster.name: bigdata-cluster\nnetwork.host: 127.0.0.1\nhttp.port: 9200\ndiscovery.type: single-node\nxpack.security.enabled: false\n")

print("Iniciando Elasticsearch...")
# ES_JAVA_OPTS cruciais: bypass cgroup e permitir security manager
os.system(f'sudo -u daemon ES_JAVA_OPTS="-Xms512m -Xmx512m -Des.monitor.cgroup.disabled=true -Djava.security.manager=allow" {ES_PATH}/bin/elasticsearch -d')

# Loop de espera
for i in range(35):
    if os.system('curl -s http://localhost:9200 > /dev/null') == 0:
        print("\nElasticsearch ONLINE!")
        break
    print(".", end="")
    time.sleep(5)

Instalando dependências...
Iniciando Elasticsearch...
...................................

In [ ]:
# --- 2. GERAÇÃO DE DADOS ---
random.seed(42)
CATEGORIAS = ['Eletrônicos', 'Roupas', 'Casa', 'Livros', 'Esportes']
os.makedirs('dados', exist_ok=True)

# Geração de Produtos
produtos_list = []
for i in range(1, 1001):
    produtos_list.append([i, f"Produto_{i}", random.choice(CATEGORIAS), round(random.uniform(10, 5000), 2)])

with open('dados/produtos.txt', 'w', newline='') as f:
    writer = csv.writer(f, delimiter=';')
    writer.writerow(['id', 'nome', 'categoria', 'preco'])
    writer.writerows(produtos_list)

# Geração de Transações
transacoes_list = []
for i in range(1, 5001):
    p = random.choice(produtos_list)
    transacoes_list.append([i, p[0], random.randint(1, 5), p[3],
                           (datetime.now() - timedelta(days=random.randint(0, 30))).strftime('%Y-%m-%d'),
                           random.randint(1, 100)])

with open('dados/transacoes.txt', 'w', newline='') as f:
    writer = csv.writer(f, delimiter=';')
    writer.writerow(['id', 'id_produto', 'quantidade', 'preco_unitario', 'data', 'id_cliente'])
    writer.writerows(transacoes_list)

print("Dados gerados com sucesso em 'dados/'.")

Dados gerados com sucesso em 'dados/'.


In [ ]:
# --- 3. PIPELINE INTEGRADO E RECOMENDAÇÃO ---

# A. Monitoramento Prometheus
try:
    from prometheus_client import start_http_server, Counter
    # Tenta iniciar na porta 8000, ignora se já estiver rodando
    try:
        start_http_server(8000)
        print("Servidor Prometheus iniciado na porta 8000.")
    except:
        pass
    vendas_metric = Counter('vendas_total', 'Total de transações processadas')
except Exception as e:
    print(f"Aviso no Prometheus: {e}")

# B. Indexação NoSQL (Elasticsearch) - Tratamento Resiliente
from elasticsearch import Elasticsearch, helpers
es = Elasticsearch(["http://localhost:9200"])

try:
    # Timeout curto para não travar a célula se o serviço estiver offline
    if es.ping():
        print("Conectado ao Elasticsearch. Indexando produtos...")
        actions = [{"_index": "loja", "_id": p[0], "_source": {"nome": p[1], "cat": p[2], "preco": p[3]}} for p in produtos_list]
        helpers.bulk(es, actions)
        print("Indexação NoSQL concluída.")
    else:
        print("[AVISO] Elasticsearch offline. Prosseguindo com Spark e Recomendações...")
except Exception as e:
    print(f"[AVISO NoSQL] Falha ao conectar no Elasticsearch: {e}")

# C. Processamento Spark
print("\n--- Análise Spark ---")
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as _sum

spark = SparkSession.builder.appName("AnaliseVendas").getOrCreate()
sdf = spark.read.option("delimiter", ";").option("header", "true").csv("dados/transacoes.txt")
sdf = sdf.withColumn("total", col("quantidade").cast("float") * col("preco_unitario").cast("float"))

print("Top 5 Produtos por Faturamento:")
sdf.groupBy("id_produto").agg(_sum("total").alias("receita")).orderBy(col("receita").desc()).show(5)

# D. Sistema de Recomendação
print("--- Sistema de Recomendação ---")
from collections import defaultdict
compras_cliente = defaultdict(list)
for t in transacoes_list:
    compras_cliente[t[5]].append(t[1])

co_ocorrencia = defaultdict(lambda: defaultdict(int))
for itens in compras_cliente.values():
    for i in range(len(itens)):
        for j in range(i + 1, len(itens)):
            co_ocorrencia[itens[i]][itens[j]] += 1
            co_ocorrencia[itens[j]][itens[i]] += 1

def recomendar(id_cliente, top_n=3):
    historico = compras_cliente.get(id_cliente, [])
    scores = defaultdict(int)
    for item in historico:
        for outro, peso in co_ocorrencia[item].items():
            if outro not in historico:
                scores[outro] += peso
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_n]

print(f"Recomendações para o cliente 84: {recomendar(84)}")

# E. Incrementar métrica
try:
    vendas_metric.inc(sdf.count())
except:
    pass

### Conclusão do Modelo Unificado
O modelo acima demonstra a integração funcional de todas as tecnologias solicitadas em um fluxo contínuo.

In [ ]:
!cat /var/log/elasticsearch/elasticsearch.log | tail -n 30

cat: /var/log/elasticsearch/elasticsearch.log: No such file or directory


In [ ]:
%%bash
# 1. Matar qualquer processo travado do Elasticsearch
pkill -f elasticsearch
sleep 2

# 2. Limpar dados antigos/bloqueados que impedem a inicialização
rm -rf /var/lib/elasticsearch/*

# 3. Criar usuário dedicado (o Colab usa root por padrão, o que quebra o ES)
useradd -m elasticuser 2>/dev/null || true

# 4. Criar um arquivo de configuração limpo e minimalista
cat <<EOF > /etc/elasticsearch/elasticsearch.yml
cluster.name: colab-cluster
node.name: colab-node
path.data: /var/lib/elasticsearch
path.logs: /var/log/elasticsearch
network.host: 127.0.0.1
http.port: 9200
discovery.type: single-node
xpack.security.enabled: false
xpack.ml.enabled: false
ingest.geoip.downloader.enabled: false
EOF

# 5. Aplicar permissões rigorosas para o novo usuário
chown -R elasticuser:elasticuser /usr/share/elasticsearch /var/lib/elasticsearch /var/log/elasticsearch /etc/elasticsearch

# 6. Iniciar o serviço em background com memória limitada
echo "Iniciando Elasticsearch..."
sudo -u elasticuser ES_JAVA_OPTS="-Xms400m -Xmx400m" /usr/share/elasticsearch/bin/elasticsearch -d

# 7. Monitorar a porta 9200 para confirmar o sucesso (timeout de 35 segundos)
for i in {1..35}; do
  if curl -s http://localhost:9200 > /dev/null; then
    echo "Elasticsearch está ONLINE e pronto para receber conexões!"
    exit 0
  fi
  sleep 1
done

echo "Falha ao iniciar. Últimas linhas do log de erro:"
cat /var/log/elasticsearch/colab-cluster.log | tail -n 40

Iniciando Elasticsearch...
Falha ao iniciar. Últimas linhas do log de erro:


bash: line 12: /etc/elasticsearch/elasticsearch.yml: No such file or directory
chown: cannot access '/usr/share/elasticsearch': No such file or directory
chown: cannot access '/var/lib/elasticsearch': No such file or directory
chown: cannot access '/var/log/elasticsearch': No such file or directory
chown: cannot access '/etc/elasticsearch': No such file or directory
sudo: /usr/share/elasticsearch/bin/elasticsearch: command not found
cat: /var/log/elasticsearch/colab-cluster.log: No such file or directory


In [ ]:
!pip uninstall -yq elastic-transport elasticsearch
!pip install -q pyspark prometheus_client elasticsearch==7.17.9 "urllib3<2.0.0"

In [ ]:
%%bash
# 1. Matar processos travados e remover a instalação com erro do apt
pkill -f elasticsearch
apt-get remove --purge -y elasticsearch > /dev/null 2>&1
rm -rf /content/elasticsearch-*

# 2. Baixar os binários standalone da versão mais recente (resolve o bug do Java Security)
cd /content
echo "Baixando Elasticsearch..."
wget -q https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-8.15.0-linux-x86_64.tar.gz
tar -xzf elasticsearch-8.15.0-linux-x86_64.tar.gz

# 3. Aplicar configurações limpas para o Colab
cat <<EOF > /content/elasticsearch-8.15.0/config/elasticsearch.yml
cluster.name: colab-cluster
node.name: colab-node
network.host: 127.0.0.1
http.port: 9200
discovery.type: single-node
xpack.security.enabled: false
xpack.ml.enabled: false
EOF

# 4. Dar permissão total ao usuário daemon apenas nesta pasta
chown -R daemon:daemon /content/elasticsearch-8.15.0

# 5. Iniciar o serviço em background
echo "Iniciando Elasticsearch..."
sudo -u daemon ES_JAVA_OPTS="-Xms400m -Xmx400m" /content/elasticsearch-8.15.0/bin/elasticsearch -d

# 6. Monitorar a porta 9200 para confirmação (timeout de 45 segundos)
for i in {1..45}; do
  if curl -s http://localhost:9200 > /dev/null; then
    echo "Elasticsearch está ONLINE e pronto para uso!"
    exit 0
  fi
  sleep 1
done

echo "Falha ao iniciar. Últimas linhas do log:"
cat /content/elasticsearch-8.15.0/logs/colab-cluster.log | tail -n 30

In [ ]:
import random
import csv
from datetime import datetime, timedelta
import os

# Sementes para reprodutibilidade
random.seed(42)

# Categorias e nomes de produtos
CATEGORIAS = ['Eletrônicos', 'Roupas', 'Casa e Decoração', 'Livros', 'Brinquedos', 'Esportes', 'Beleza', 'Automotivo']
PRODUTOS_POR_CATEGORIA = {
    'Eletrônicos': ['Smartphone', 'Notebook', 'Tablet', 'Fone de Ouvido', 'Carregador', 'Monitor', 'Teclado', 'Mouse'],
    'Roupas': ['Camiseta', 'Calça Jeans', 'Vestido', 'Jaqueta', 'Sapato', 'Bolsa', 'Cinto'],
    'Casa e Decoração': ['Sofá', 'Mesa', 'Cadeira', 'Estante', 'Luminária', 'Quadro', 'Tapete'],
    'Livros': ['Romance', 'Ficção Científica', 'Biografia', 'Autoajuda', 'Fantasia', 'Suspense'],
    'Brinquedos': ['Boneca', 'Carrinho', 'Quebra-cabeça', 'Jogo de Tabuleiro', 'Pelúcia'],
    'Esportes': ['Bola', 'Tênis', 'Raquete', 'Bicicleta', 'Halter'],
    'Beleza': ['Shampoo', 'Condicionador', 'Base', 'Máscara Facial', 'Perfume'],
    'Automotivo': ['Pneu', 'Bateria', 'Óleo', 'Filtro', 'Palheta']
}

# Função para gerar nome de produto
def gerar_nome_produto(categoria):
    return random.choice(PRODUTOS_POR_CATEGORIA[categoria]) + ' ' + str(random.randint(1, 100))

# Geração de produtos
def gerar_produtos(qtd=10000):
    produtos = []
    for i in range(1, qtd+1):
        categoria = random.choice(CATEGORIAS)
        nome = gerar_nome_produto(categoria)
        preco = round(random.uniform(10.0, 5000.0), 2)
        descricao = f'{nome} de alta qualidade, ideal para uso diário.'
        estoque = random.randint(0, 100)
        produtos.append([i, nome, categoria, preco, descricao, estoque])
    return produtos

# Geração de avaliações
def gerar_avaliacoes(produtos, qtd=50000):
    avaliacoes = []
    ids_produtos = [p[0] for p in produtos]
    for i in range(1, qtd+1):
        id_prod = random.choice(ids_produtos)
        nota = random.randint(1, 5)
        comentario = random.choice(['Ótimo!', 'Bom produto.', 'Poderia ser melhor.', 'Excelente!', 'Não gostei.'])
        data = datetime.now() - timedelta(days=random.randint(1, 730))
        avaliacoes.append([i, id_prod, nota, comentario, data.strftime('%Y-%m-%d')])
    return avaliacoes

# Geração de transações
def gerar_transacoes(produtos, qtd=100000):
    transacoes = []
    ids_produtos = [p[0] for p in produtos]
    for i in range(1, qtd+1):
        id_prod = random.choice(ids_produtos)
        preco_unit = next(p[3] for p in produtos if p[0] == id_prod)
        quantidade = random.randint(1, 5)
        data = datetime.now() - timedelta(days=random.randint(1, 365))
        id_cliente = random.randint(1, 5000)
        transacoes.append([i, id_prod, quantidade, preco_unit, data.strftime('%Y-%m-%d'), id_cliente])
    return transacoes

# Salvar dados em arquivos .txt (CSV)
def salvar_dados(dados, nome_arquivo, cabecalho):
    with open(nome_arquivo, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f, delimiter=';')
        writer.writerow(cabecalho)
        writer.writerows(dados)

# Criar diretório para dados
os.makedirs('dados', exist_ok=True)

# Gerar e salvar
produtos = gerar_produtos(10000)
salvar_dados(produtos, 'dados/produtos.txt', ['id', 'nome', 'categoria', 'preco', 'descricao', 'estoque'])

avaliacoes = gerar_avaliacoes(produtos, 50000)
salvar_dados(avaliacoes, 'dados/avaliacoes.txt', ['id', 'id_produto', 'nota', 'comentario', 'data'])

transacoes = gerar_transacoes(produtos, 100000)
salvar_dados(transacoes, 'dados/transacoes.txt', ['id', 'id_produto', 'quantidade', 'preco_unitario', 'data', 'id_cliente'])

# Particionamento por categoria (sharding) - exemplo para produtos
for categoria in CATEGORIAS:
    produtos_cat = [p for p in produtos if p[2] == categoria]
    if produtos_cat:
        salvar_dados(produtos_cat, f'dados/produtos_{categoria}.txt', ['id', 'nome', 'categoria', 'preco', 'descricao', 'estoque'])

# Particionamento por mês (sharding) - exemplo para transações
from collections import defaultdict
transacoes_por_mes = defaultdict(list)
for t in transacoes:
    mes = t[4][:7]  # ano-mês
    transacoes_por_mes[mes].append(t)

for mes, lista in transacoes_por_mes.items():
    salvar_dados(lista, f'dados/transacoes_{mes}.txt', ['id', 'id_produto', 'quantidade', 'preco_unitario', 'data', 'id_cliente'])

print("Dados gerados com sucesso!")

Dados gerados com sucesso!


In [ ]:
from elasticsearch import Elasticsearch, helpers
import json

# Conectar ao Elasticsearch (localhost:9200)
es = Elasticsearch([{'host': 'localhost', 'port': 9200}])

if es.ping():
    print("Conectado ao Elasticsearch")
    # Criar índice
    es.indices.create(index='produtos', ignore=400)
    # Indexar produtos (usando bulk)
    actions = []
    for p in produtos:
        doc = {
            '_index': 'produtos',
            '_id': p[0],
            '_source': {
                'nome': p[1],
                'categoria': p[2],
                'preco': p[3],
                'descricao': p[4],
                'estoque': p[5]
            }
        }
        actions.append(doc)
    helpers.bulk(es, actions)
    print("Produtos indexados.")

    # Busca por "Smartphone" na categoria "Eletrônicos"
    res = es.search(index='produtos', body={
        "query": {
            "bool": {
                "must": [
                    {"match": {"nome": "Smartphone"}},
                    {"term": {"categoria": "Eletrônicos"}}
                ]
            }
        }
    })
    print("Resultados da busca:")
    for hit in res['hits']['hits']:
        print(hit['_source'])
else:
    print("Elasticsearch não disponível. Usando simulação em memória.")

In [ ]:
import pandas as pd
import glob

# Lê todos os arquivos de transações particionados por mês
arquivos = glob.glob('dados/transacoes_*.txt')
dfs = []
for arq in arquivos:
    df = pd.read_csv(arq, delimiter=';')
    dfs.append(df)
df_transacoes = pd.concat(dfs, ignore_index=True)

# Lê produtos
df_produtos = pd.read_csv('dados/produtos.txt', delimiter=';')

# Junta as tabelas
df_vendas = df_transacoes.merge(df_produtos[['id', 'categoria']], left_on='id_produto', right_on='id')
df_vendas['faturamento'] = df_vendas['quantidade'] * df_vendas['preco_unitario']
df_vendas['mes'] = pd.to_datetime(df_vendas['data']).dt.to_period('M')

# Agregação por categoria e mês
agregado = df_vendas.groupby(['categoria', 'mes']).agg(
    total_vendas=('faturamento', 'sum'),
    qtde_itens=('quantidade', 'sum')
).reset_index()

print(agregado.head(10))

# Salvar resultado
agregado.to_csv('dados/analise_vendas.txt', sep=';', index=False)

    categoria      mes  total_vendas  qtde_itens
0  Automotivo  2025-09    6993214.41        2795
1  Automotivo  2025-10    7746701.96        3178
2  Automotivo  2025-11    7435670.26        3033
3  Automotivo  2025-12    7355819.91        3115
4  Automotivo  2026-01    7476706.75        3153
5  Automotivo  2026-02    6743753.94        2805
6  Automotivo  2026-03    7714916.64        3169
7  Automotivo  2026-04    8018068.66        3278
8  Automotivo  2026-05    7689939.29        3069
9  Automotivo  2026-06    7311527.39        3031


In [ ]:
from collections import defaultdict

# Carregar transações
df_transacoes = pd.read_csv('dados/transacoes.txt', delimiter=';')

# Criar lista de compras por cliente
compras_cliente = defaultdict(list)
for _, row in df_transacoes.iterrows():
    compras_cliente[row['id_cliente']].append(row['id_produto'])

# Calcular co-ocorrência de produtos (quem comprou A também comprou B)
co_ocorrencia = defaultdict(lambda: defaultdict(int))
for produtos_comprados in compras_cliente.values():
    for i in range(len(produtos_comprados)):
        for j in range(i+1, len(produtos_comprados)):
            a, b = produtos_comprados[i], produtos_comprados[j]
            co_ocorrencia[a][b] += 1
            co_ocorrencia[b][a] += 1

# Função de recomendação para um cliente
def recomendar_para_cliente(id_cliente, top_n=5):
    if id_cliente not in compras_cliente:
        return []
    historico = compras_cliente[id_cliente]
    # Soma pontuações dos produtos co-ocorrentes
    scores = defaultdict(float)
    for item in historico:
        for outro, contagem in co_ocorrencia[item].items():
            if outro not in historico:  # não recomendar já comprados
                scores[outro] += contagem
    # Ordenar
    recomendados = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [prod_id for prod_id, score in recomendados[:top_n]]

# Exemplo para cliente 123
cliente_exemplo = 123
recs = recomendar_para_cliente(cliente_exemplo)
print(f"Recomendações para cliente {cliente_exemplo}:")
for prod_id in recs:
    produto = df_produtos[df_produtos['id'] == prod_id]
    if not produto.empty:
        print(produto.iloc[0]['nome'])

Recomendações para cliente 123:
Óleo 80
Fantasia 30
Jogo de Tabuleiro 94
Bateria 44
Estante 49


In [ ]:
"""
ecommerce_bigdata_demo.py
Exemplo completo: gera dados fictícios, insere no Elasticsearch, processa com Spark,
faz buscas e recomendações simples.
"""

import random
import time
import json
from datetime import datetime, timedelta

import pandas as pd

# Elasticsearch client
from elasticsearch import Elasticsearch, helpers

# PySpark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, desc

# ---------------------------
# Configurações
# ---------------------------
ES_HOST = "localhost"
ES_PORT = 9200
INDEX_NAME = "ecommerce"
BULK_CHUNK = 500  # tamanho do chunk para bulk insert

# ---------------------------
# 1) Gerar dados fictícios
# ---------------------------
def generate_products():
    produtos = [
        {"id": 1, "nome": "Notebook Dell", "categoria": "Eletrônicos", "preco": 3500.0, "marca": "Dell"},
        {"id": 2, "nome": "Smartphone Samsung", "categoria": "Eletrônicos", "preco": 2500.0, "marca": "Samsung"},
        {"id": 3, "nome": "Geladeira Brastemp", "categoria": "Eletrodomésticos", "preco": 1800.0, "marca": "Brastemp"},
        {"id": 4, "nome": "TV LG 50\"", "categoria": "Eletrônicos", "preco": 2800.0, "marca": "LG"},
        {"id": 5, "nome": "Cafeteira Nespresso", "categoria": "Eletrodomésticos", "preco": 600.0, "marca": "Nespresso"},
        {"id": 6, "nome": "Fone Bluetooth JBL", "categoria": "Eletrônicos", "preco": 350.0, "marca": "JBL"},
        {"id": 7, "nome": "Micro-ondas Electrolux", "categoria": "Eletrodomésticos", "preco": 700.0, "marca": "Electrolux"},
    ]
    return produtos

def generate_reviews(products, n_reviews=50):
    usuarios = [f"user_{i}" for i in range(1, 31)]
    comentarios = [
        "Excelente produto", "Bom custo-benefício", "Recomendo", "Não gostei", "Entrega rápida",
        "Qualidade razoável", "Muito satisfeito", "Poderia ser melhor"
    ]
    reviews = []
    for _ in range(n_reviews):
        p = random.choice(products)
        reviews.append({
            "produto_id": p["id"],
            "usuario": random.choice(usuarios),
            "nota": random.randint(1, 5),
            "comentario": random.choice(comentarios),
            "data": (datetime.now() - timedelta(days=random.randint(0, 365))).isoformat()
        })
    return reviews

def generate_transactions(products, n_tx=200):
    transacoes = []
    usuarios = [f"cliente_{i}" for i in range(1, 101)]
    for i in range(1, n_tx + 1):
        produto = random.choice(products)
        quantidade = random.randint(1, 4)
        valor_unit = produto["preco"] * random.uniform(0.8, 1.2)
        valor_total = round(valor_unit * quantidade, 2)
        transacoes.append({
            "transacao_id": i,
            "produto_id": produto["id"],
            "usuario": random.choice(usuarios),
            "quantidade": quantidade,
            "valor_total": valor_total,
            "data": (datetime.now() - timedelta(days=random.randint(0, 90))).isoformat()
        })
    return transacoes

# ---------------------------
# 2) Conectar e inserir no Elasticsearch
# ---------------------------
def connect_es(host=ES_HOST, port=ES_PORT):
    es = Elasticsearch([{"host": host, "port": port}])
    if not es.ping():
        raise ConnectionError(f"Não foi possível conectar ao Elasticsearch em {host}:{port}")
    return es

def create_index_if_not_exists(es, index_name=INDEX_NAME):
    # Mapeamento simples para demonstrar full-text e campos numéricos
    mapping = {
        "mappings": {
            "properties": {
                "nome": {"type": "text"},
                "categoria": {"type": "keyword"},
                "marca": {"type": "keyword"},
                "preco": {"type": "double"},
                "nota": {"type": "integer"},
                "comentario": {"type": "text"},
                "data": {"type": "date"},
                "valor_total": {"type": "double"},
                "quantidade": {"type": "integer"}
            }
        }
    }
    if not es.indices.exists(index=index_name):
        es.indices.create(index=index_name, body=mapping)

def bulk_insert_products(es, products, index_name=INDEX_NAME):
    actions = [
        {"_index": index_name, "_id": f"product_{p['id']}", "_source": p}
        for p in products
    ]
    helpers.bulk(es, actions, chunk_size=BULK_CHUNK)

def bulk_insert_reviews(es, reviews, index_name=INDEX_NAME):
    actions = [
        {"_index": index_name, "_source": r}
        for r in reviews
    ]
    helpers.bulk(es, actions, chunk_size=BULK_CHUNK)

def bulk_insert_transactions(es, transactions, index_name=INDEX_NAME):
    actions = [
        {"_index": index_name, "_source": t}
        for t in transactions
    ]
    helpers.bulk(es, actions, chunk_size=BULK_CHUNK)

# ---------------------------
# 3) Buscas com Query DSL (Exemplos)
# ---------------------------
def search_by_category(es, category, index_name=INDEX_NAME, size=10):
    body = {
        "query": {
            "match": {"categoria": category}
        },
        "size": size
    }
    res = es.search(index=index_name, body=body)
    return [hit["_source"] for hit in res["hits"]["hits"]]

def search_fulltext(es, text, index_name=INDEX_NAME, size=10):
    body = {
        "query": {
            "multi_match": {
                "query": text,
                "fields": ["nome^3", "comentario", "marca"]
            }
        },
        "size": size
    }
    res = es.search(index=index_name, body=body)
    return [hit["_source"] for hit in res["hits"]["hits"]]

# ---------------------------
# 4) Processamento com Spark
# ---------------------------
def spark_session(app_name="EcommerceSparkApp"):
    spark = SparkSession.builder \
        .appName(app_name) \
        .master("local[*]") \
        .config("spark.ui.showConsoleProgress", "false") \
        .getOrCreate()
    return spark

def analyze_transactions_with_spark(transactions):
    spark = spark_session()
    df = spark.createDataFrame(transactions)
    # Total de vendas por produto
    vendas_por_produto = df.groupBy("produto_id").agg(spark_sum("valor_total").alias("total_vendas"))
    vendas_por_produto = vendas_por_produto.orderBy(desc("total_vendas"))
    vendas_por_produto.show(truncate=False)
    # Produto mais vendido por quantidade
    mais_vendido = df.groupBy("produto_id").agg(spark_sum("quantidade").alias("qtd_vendida")).orderBy(desc("qtd_vendida"))
    mais_vendido.show(truncate=False)
    # Coletar resultados para uso posterior (pegar top 3)
    top3 = mais_vendido.limit(3).toPandas()
    spark.stop()
    return top3

# ---------------------------
# 5) Recomendação simples (combina vendas e avaliações)
# ---------------------------
def recommend_product(es, transactions, reviews, top_n=3):
    # Recomendação simples: produtos com maior soma de quantidade + média de nota
    df_tx = pd.DataFrame(transactions)
    df_rev = pd.DataFrame(reviews)
    vendas = df_tx.groupby("produto_id")["quantidade"].sum().rename("total_qtd")
    medias = df_rev.groupby("produto_id")["nota"].mean().rename("media_nota")
    score = pd.concat([vendas, medias], axis=1).fillna(0)
    # Normalizar e combinar
    score["score"] = (score["total_qtd"] / (score["total_qtd"].max() or 1)) * 0.7 + (score["media_nota"] / 5.0) * 0.3
    top = score.sort_values("score", ascending=False).head(top_n).reset_index()
    # Buscar detalhes no Elasticsearch
    recs = []
    for _, row in top.iterrows():
        prod = es.get(index=INDEX_NAME, id=f"product_{int(row['produto_id'])}")["_source"]
        recs.append({"produto": prod, "score": float(row["score"])})
    return recs

# ---------------------------
# 6) Fluxo principal
# ---------------------------
def main():
    print("Gerando dados fictícios...")
    products = generate_products()
    reviews = generate_reviews(products, n_reviews=120)
    transactions = generate_transactions(products, n_tx=800)

    print("Conectando ao Elasticsearch...")
    es = connect_es()

    print("Criando índice (se necessário) e inserindo dados...")
    create_index_if_not_exists(es, INDEX_NAME)
    # Inserir produtos com IDs estáveis
    bulk_insert_products(es, products)
    # Inserir reviews e transações (como documentos separados)
    bulk_insert_reviews(es, reviews)
    bulk_insert_transactions(es, transactions)

    print("Pausando brevemente para garantir indexação...")
    time.sleep(2)

    print("\nExemplo de busca por categoria 'Eletrônicos':")
    eletr = search_by_category(es, "Eletrônicos", size=5)
    for p in eletr:
        print(f" - {p.get('nome')} | R$ {p.get('preco')}")

    print("\nExemplo de busca full-text por 'Notebook':")
    ft = search_fulltext(es, "Notebook", size=5)
    for p in ft:
        print(f" - {p.get('nome')} | Categoria: {p.get('categoria')}")

    print("\nAnalisando transações com Spark (total vendas e mais vendidos)...")
    top3 = analyze_transactions_with_spark(transactions)
    print("\nTop 3 produtos por quantidade vendidos (Spark):")
    print(top3)

    print("\nGerando recomendações simples combinando vendas e avaliações...")
    recs = recommend_product(es, transactions, reviews, top_n=3)
    for r in recs:
        prod = r["produto"]
        print(f"Recomendado: {prod['nome']} (categoria: {prod['categoria']}) - score: {r['score']:.3f}")

    print("\nFluxo concluído.")

if __name__ == "__main__":
    main()


In [ ]:
import random
import csv
from datetime import datetime, timedelta
import os
import pandas as pd
from collections import defaultdict
import glob

# ------------------------
# 1. Geração de dados fictícios
# ------------------------
random.seed(42)

CATEGORIAS = ['Eletrônicos', 'Roupas', 'Casa e Decoração', 'Livros', 'Brinquedos', 'Esportes', 'Beleza', 'Automotivo']
PRODUTOS_POR_CATEGORIA = {
    'Eletrônicos': ['Smartphone', 'Notebook', 'Tablet', 'Fone de Ouvido', 'Carregador', 'Monitor', 'Teclado', 'Mouse'],
    'Roupas': ['Camiseta', 'Calça Jeans', 'Vestido', 'Jaqueta', 'Sapato', 'Bolsa', 'Cinto'],
    'Casa e Decoração': ['Sofá', 'Mesa', 'Cadeira', 'Estante', 'Luminária', 'Quadro', 'Tapete'],
    'Livros': ['Romance', 'Ficção Científica', 'Biografia', 'Autoajuda', 'Fantasia', 'Suspense'],
    'Brinquedos': ['Boneca', 'Carrinho', 'Quebra-cabeça', 'Jogo de Tabuleiro', 'Pelúcia'],
    'Esportes': ['Bola', 'Tênis', 'Raquete', 'Bicicleta', 'Halter'],
    'Beleza': ['Shampoo', 'Condicionador', 'Base', 'Máscara Facial', 'Perfume'],
    'Automotivo': ['Pneu', 'Bateria', 'Óleo', 'Filtro', 'Palheta']
}

def gerar_nome_produto(categoria):
    return random.choice(PRODUTOS_POR_CATEGORIA[categoria]) + ' ' + str(random.randint(1, 100))

def gerar_produtos(qtd=10000):
    produtos = []
    for i in range(1, qtd+1):
        categoria = random.choice(CATEGORIAS)
        nome = gerar_nome_produto(categoria)
        preco = round(random.uniform(10.0, 5000.0), 2)
        descricao = f'{nome} de alta qualidade, ideal para uso diário.'
        estoque = random.randint(0, 100)
        produtos.append([i, nome, categoria, preco, descricao, estoque])
    return produtos

def gerar_avaliacoes(produtos, qtd=50000):
    avaliacoes = []
    ids_produtos = [p[0] for p in produtos]
    for i in range(1, qtd+1):
        id_prod = random.choice(ids_produtos)
        nota = random.randint(1, 5)
        comentario = random.choice(['Ótimo!', 'Bom produto.', 'Poderia ser melhor.', 'Excelente!', 'Não gostei.'])
        data = datetime.now() - timedelta(days=random.randint(1, 730))
        avaliacoes.append([i, id_prod, nota, comentario, data.strftime('%Y-%m-%d')])
    return avaliacoes

def gerar_transacoes(produtos, qtd=100000):
    transacoes = []
    ids_produtos = [p[0] for p in produtos]
    for i in range(1, qtd+1):
        id_prod = random.choice(ids_produtos)
        preco_unit = next(p[3] for p in produtos if p[0] == id_prod)
        quantidade = random.randint(1, 5)
        data = datetime.now() - timedelta(days=random.randint(1, 365))
        id_cliente = random.randint(1, 5000)
        transacoes.append([i, id_prod, quantidade, preco_unit, data.strftime('%Y-%m-%d'), id_cliente])
    return transacoes

def salvar_dados(dados, nome_arquivo, cabecalho):
    with open(nome_arquivo, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f, delimiter=';')
        writer.writerow(cabecalho)
        writer.writerows(dados)

os.makedirs('dados', exist_ok=True)

produtos = gerar_produtos(10000)
salvar_dados(produtos, 'dados/produtos.txt', ['id', 'nome', 'categoria', 'preco', 'descricao', 'estoque'])

avaliacoes = gerar_avaliacoes(produtos, 50000)
salvar_dados(avaliacoes, 'dados/avaliacoes.txt', ['id', 'id_produto', 'nota', 'comentario', 'data'])

transacoes = gerar_transacoes(produtos, 100000)
salvar_dados(transacoes, 'dados/transacoes.txt', ['id', 'id_produto', 'quantidade', 'preco_unitario', 'data', 'id_cliente'])

# Particionamento (sharding) por categoria
for categoria in CATEGORIAS:
    produtos_cat = [p for p in produtos if p[2] == categoria]
    if produtos_cat:
        salvar_dados(produtos_cat, f'dados/produtos_{categoria}.txt', ['id', 'nome', 'categoria', 'preco', 'descricao', 'estoque'])

# Particionamento por mês
transacoes_por_mes = defaultdict(list)
for t in transacoes:
    mes = t[4][:7]
    transacoes_por_mes[mes].append(t)

for mes, lista in transacoes_por_mes.items():
    salvar_dados(lista, f'dados/transacoes_{mes}.txt', ['id', 'id_produto', 'quantidade', 'preco_unitario', 'data', 'id_cliente'])

print("Arquivos .txt gerados em 'dados/'.")

# ------------------------
# 2. Busca distribuída simulada (índice invertido)
# ------------------------
class IndiceInvertido:
    def __init__(self):
        self.indice = {}

    def indexar_documento(self, doc_id, texto, categoria=None):
        palavras = set(texto.lower().split())
        for palavra in palavras:
            if palavra not in self.indice:
                self.indice[palavra] = set()
            self.indice[palavra].add(doc_id)

    def buscar(self, termo, categoria=None):
        termo = termo.lower()
        ids = self.indice.get(termo, set())
        if categoria:
            ids = {id_ for id_ in ids if produtos_dict[id_][2] == categoria}
        return ids

# Construir índice com nome e descrição
indice = IndiceInvertido()
produtos_dict = {p[0]: p for p in produtos}
for p in produtos:
    indice.indexar_documento(p[0], p[1] + ' ' + p[4], p[2])

print("\n--- Busca distribuída (simulada) ---")
resultados = indice.buscar("smartphone", categoria="Eletrônicos")
print(f"IDs encontrados para 'smartphone' em Eletrônicos: {resultados}")
for id_ in list(resultados)[:5]:
    print(f"  {produtos_dict[id_][1]} (R$ {produtos_dict[id_][3]:.2f})")

# ------------------------
# 3. Pipeline de análise (pandas)
# ------------------------
arquivos = glob.glob('dados/transacoes_*.txt')
dfs = []
for arq in arquivos:
    df = pd.read_csv(arq, delimiter=';')
    dfs.append(df)
df_transacoes = pd.concat(dfs, ignore_index=True)

df_produtos = pd.read_csv('dados/produtos.txt', delimiter=';')

df_vendas = df_transacoes.merge(df_produtos[['id', 'categoria']], left_on='id_produto', right_on='id')
df_vendas['faturamento'] = df_vendas['quantidade'] * df_vendas['preco_unitario']
df_vendas['mes'] = pd.to_datetime(df_vendas['data']).dt.to_period('M')

agregado = df_vendas.groupby(['categoria', 'mes']).agg(
    total_vendas=('faturamento', 'sum'),
    qtde_itens=('quantidade', 'sum')
).reset_index()

print("\n--- Pipeline de análise (vendas por categoria/mês) ---")
print(agregado.head(10))
agregado.to_csv('dados/analise_vendas.txt', sep=';', index=False)

# ------------------------
# 4. Recomendações personalizadas (co-ocorrência)
# ------------------------
compras_cliente = defaultdict(list)
for _, row in df_transacoes.iterrows():
    compras_cliente[row['id_cliente']].append(row['id_produto'])

co_ocorrencia = defaultdict(lambda: defaultdict(int))
for produtos_comprados in compras_cliente.values():
    for i in range(len(produtos_comprados)):
        for j in range(i+1, len(produtos_comprados)):
            a, b = produtos_comprados[i], produtos_comprados[j]
            co_ocorrencia[a][b] += 1
            co_ocorrencia[b][a] += 1

def recomendar_para_cliente(id_cliente, top_n=5):
    if id_cliente not in compras_cliente:
        return []
    historico = compras_cliente[id_cliente]
    scores = defaultdict(float)
    for item in historico:
        for outro, contagem in co_ocorrencia[item].items():
            if outro not in historico:
                scores[outro] += contagem
    recomendados = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [prod_id for prod_id, _ in recomendados[:top_n]]

cliente_exemplo = 123
recs = recomendar_para_cliente(cliente_exemplo)
print(f"\n--- Recomendações para cliente {cliente_exemplo} ---")
for prod_id in recs:
    produto = df_produtos[df_produtos['id'] == prod_id]
    if not produto.empty:
        print(f"  {produto.iloc[0]['nome']} (categoria: {produto.iloc[0]['categoria']})")

Arquivos .txt gerados em 'dados/'.

--- Busca distribuída (simulada) ---
IDs encontrados para 'smartphone' em Eletrônicos: {9216, 1537, 4609, 522, 523, 9228, 7693, 1039, 2581, 7189, 4633, 5661, 7709, 5149, 6686, 3625, 9775, 2609, 1590, 4665, 4162, 3139, 6211, 6217, 3148, 1103, 4177, 1106, 1623, 5721, 9817, 9823, 4704, 6753, 8292, 9316, 7783, 3696, 9328, 5747, 5236, 8312, 4216, 2172, 129, 130, 7306, 8331, 3214, 6286, 4754, 3222, 4760, 5785, 1178, 674, 5798, 3244, 9901, 8370, 3255, 1723, 8381, 4801, 4803, 4811, 9422, 5331, 4322, 742, 8940, 4334, 239, 248, 4858, 6907, 4348, 5372, 8958, 9984, 6402, 1300, 6420, 1815, 5918, 3362, 7460, 2341, 4902, 9512, 6446, 6960, 5426, 6968, 7996, 5953, 2882, 1359, 6488, 9049, 2908, 1373, 4446, 5982, 865, 1388, 7028, 7040, 3458, 1923, 8584, 9096, 6034, 2454, 5526, 5537, 421, 5546, 9643, 2477, 2484, 6586, 4539, 9662, 9151, 4034, 5058, 6597, 454, 967, 4553, 8138, 4555, 6606, 4561, 4050, 7125, 3035, 3041, 7650, 5094, 487, 4071, 3562, 5104, 7153, 3058, 8179, 8

### 1. Instalação de Dependências e Configuração do Elasticsearch Standalone
Como o Elasticsearch requer configurações específicas de usuário e memória que o `apt` do Colab muitas vezes bloqueia, utilizaremos a versão tarball para garantir a execução.

In [ ]:
import os
import time

# 1. Instalação
!pip install -q pyspark prometheus_client elasticsearch==6.8.2

# 2. Setup
path = "/content/elasticsearch-6.8.23"
if not os.path.exists(path):
    print("Baixando Elasticsearch 6.8.23...")
    !wget -q https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-6.8.23.tar.gz -O elasticsearch.tar.gz
    !tar -xzf elasticsearch.tar.gz
    !rm elasticsearch.tar.gz

# 3. Preparação de ambiente e permissões rigorosas
!pkill -9 -f elasticsearch
!rm -rf {path}/data/* {path}/logs/*
!mkdir -p {path}/data {path}/logs
!chmod -R 777 {path}
!chown -R daemon:daemon {path}

with open(f'{path}/config/elasticsearch.yml', 'w') as f:
    f.write("cluster.name: bigdata-cluster\n")
    f.write("network.host: 127.0.0.1\n")
    f.write("http.port: 9200\n")
    f.write("discovery.type: single-node\n")
    f.write("xpack.security.enabled: false\n")
    f.write("xpack.monitoring.enabled: false\n")
    f.write("bootstrap.memory_lock: false\n")

print("Iniciando Elasticsearch 6.8.23...")
# ES_JAVA_OPTS: desabilitando cgroup e fixando heap para estabilidade
cmd = f'sudo -u daemon ES_JAVA_OPTS="-Xms512m -Xmx512m -Des.monitor.cgroup.disabled=true -Djava.security.manager=allow" {path}/bin/elasticsearch -d'
os.system(cmd)

# Loop de verificação
print("Aguardando o serviço subir na porta 9200...")
for i in range(20):
    status = os.system('curl -s http://localhost:9200 > /dev/null')
    if status == 0:
        print("\nElasticsearch ONLINE!")
        !curl -X GET "localhost:9200/"
        break
    print(".", end="")
    time.sleep(5)
else:
    print("\nO serviço não subiu. Analisando log:")
    log_file = f"{path}/logs/bigdata-cluster.log"
    if os.path.exists(log_file):
        !tail -n 50 {log_file}
    else:
        print("Log não encontrado. Tente rodar a célula novamente.")

Iniciando Elasticsearch 6.8.23...
Aguardando o serviço subir na porta 9200...
..............................
O serviço não subiu. Verificando logs internos:
Arquivo de log não gerado. Verifique permissões do diretório.


### 2. Projeto Integrado: NoSQL + Spark + Recomendações + Monitoramento
Este script unifica todas as etapas da atividade: inserção distribuída, análise com Spark, recomendações e exposição de métricas.

In [ ]:
from elasticsearch import Elasticsearch, helpers
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as _sum
from prometheus_client import start_http_server, Counter
import pandas as pd

# --- ETAPA 5: Monitoramento ---
try:
    start_http_server(8000)
    print("Métricas Prometheus na porta 8000")
except: pass

TX_COUNTER = Counter('total_vendas_processadas', 'Total de transações')

# 1. Conexão NoSQL (Etapa 2)
es = Elasticsearch(["http://localhost:9200"])
try:
    if es.ping():
        print("[ETAPA 2] NoSQL Conectado. Indexando...")
        actions = [{
            "_index": "produtos",
            "_id": p[0],
            "_source": {"nome": p[1], "categoria": p[2], "preco": p[3]}
        } for p in produtos[:500]]
        helpers.bulk(es, actions)
        print("Busca rápida concluída.")
    else:
        print("[AVISO] Elasticsearch não respondeu. Pulando Etapa 2.")
except Exception as e:
    print(f"[ERRO NoSQL] {e}")

# 2. Pipeline de Análise com Apache Spark (Etapa 3)
print("\n[ETAPA 3] Processamento Spark...")
spark = SparkSession.builder.appName("BigDataActivity").getOrCreate()
sdf = spark.read.option("delimiter", ";").option("header", "true").csv("dados/transacoes.txt")
sdf = sdf.withColumn("total", col("quantidade").cast("float") * col("preco_unitario").cast("float"))

analise = sdf.groupBy("id_produto").agg(_sum("total").alias("faturamento"))
analise.orderBy(col("faturamento").desc()).show(5)
TX_COUNTER.inc(sdf.count())

# 3. Integração (Etapa 4)
print("\n[ETAPA 4] Recomendação Personalizada:")
# Chamando a função definida anteriormente
try:
    print(f"Cliente 123 -> {recomendar_para_cliente(123, top_n=3)}")
except NameError:
    print("Função de recomendação não encontrada. Certifique-se de executar as células anteriores.")